In [ ]:
# pip install nest-asyncio

# Creating an A2A Healthcare Provider Agent using OpenAI and MCP

In this lesson, you will build a third agent: a Healthcare Provider Agent. This agent demonstrates a powerful combination of technologies:
1.  **MCP (Model Context Protocol)**: You will build a server that exposes a tool to find doctors from a JSON file.
2.  **OpenAI**: You will build an agent that uses this MCP tool.
3.  **A2A**: You will wrap the OpenAI agent in an A2A server.

## Create the MCP Server

First, you will define an MCP server using `FastMCP`. This server exposes a tool called `list_doctors` which queries a local `doctors.json` file.

In [ ]:
%%writefile mcpserver.py
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

# Initialize the server
mcp = FastMCP("doctorserver")

# Load Data
doctors: list = json.loads(Path("../data/philippines_doctors.json").read_text())


@mcp.tool()
def list_doctors(state: str | None = None, city: str | None = None) -> list[dict]:
    """This tool returns a list of doctors practicing in a specific location. The search is case-insensitive.

    Args:
        state: The province in the Philippines.
        city: The name of the city or town (e.g., "Mandaluyong").

    Returns:
        A JSON string representing a list of doctors matching the criteria.
        If no criteria are provided, an error message is returned.
        Example: '[{"name": "Dr John James", "specialty": "Cardiology", ...}]'
    """
    # Input validation: ensure at least one search term is given.
    if not state and not city:
        return [{"error": "Please provide a state or a city."}]

    target_state = state.strip().lower() if state else None
    target_city = city.strip().lower() if city else None

    return [
        doc
        for doc in doctors
        if (not target_state or doc["address"]["state"].lower() == target_state)
        and (not target_city or doc["address"]["city"].lower() == target_city)
    ]


# Kick off server if file is run
if __name__ == "__main__":
    mcp.run(transport="stdio")


## Test with OpenAI and MCP Client

Now you will verify that you can connect to the MCP server and use it within an OpenAI agent. You use `ClientSession` to connect to the local script via `stdio` transport.

In [ ]:
%%writefile hpagent.py
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from typing import List
import asyncio
import nest_asyncio
import json
from gates_openai import create_response

nest_asyncio.apply()

class ProviderAgent:

    def __init__(self):
        # Initialize session and client objects
        self.session: ClientSession = None
        self.available_tools: List[dict] = []

    async def process_query(self, query: str = None):

        messages = [
            {
                "role": "system",
                "content": """Your task is to find a list of providers using the list_doctor MCP tool based on the user's query.
                Only use providers based on teh response from the tool.
                Output the information in a table.
                """
            },
            {
                "role":"user",
                "content":query
            }
        ]


        response = create_response(
            model = "gpt-4o-mini",
            tools = self.available_tools,
            input = messages,
        )

        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if not function_calls:
            return response.output_text, response.id
        
        tool_outputs = []

        while True:

            for call in function_calls:

                tool_name = call.name
                tool_args = json.loads(call.arguments)

                result = await self.session.call_tool(tool_name, arguments = tool_args)
                tool_output = "\n".join(map(str, result))
                # print(tool_output)

                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": tool_output
                })

            messages.extend(tool_outputs)

            response = create_response(
                model = "gpt-4o-mini",
                input = tool_outputs,
                tools = self.available_tools,
                previous_response_id = response.id
            )


            function_calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            if function_calls:
                print("Function calls found...")
                continue
            else:
                return response.output_text


    async def connect_to_server_and_run(self, query):
        # Create server parameters for stdio connection
        server_params = StdioServerParameters(
            command="uv",  # Executable
            args=["run", "mcpserver.py"],  # Optional command line arguments
            env=None,  # Optional environment variables
        )
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                self.session = session
                # Initialize the connection
                await session.initialize()

                # List available tools
                response = await session.list_tools()

                tools = response.tools
                print("\nConnected to server with tools:", [tool.name for tool in tools])

                self.available_tools = [{
                    "type": "function",
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                } for tool in response.tools]

                return await self.process_query(query)


In [ ]:
from hpagent import ProviderAgent
from IPython.display import Markdown, display

agent = ProviderAgent()

result = await agent.connect_to_server_and_run(
    "I'm based in Caloocan. Are there any Psychiatrists near me?"
)

display(Markdown(result))


## Wrap in A2A Server

Finally, create the `a2a_provider_agent.py` server file. This uses the standard A2A wrapping pattern you learned previously, but with the added complexity of asynchronous initialization for the MCP client in the `ProviderAgentExecutor`.

In [ ]:
%%writefile a2a_provider_agent.py
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)
from a2a.utils import new_agent_text_message
from hpagent import ProviderAgent

class ProviderAgentExecutor(AgentExecutor):
    """This is an agent for finding healthcare providers based on location and specialty."""
    
    def __init__(self) -> None:
        # Don't await in __init__ - it's not async
        self.agent = ProviderAgent()
    
    # async def _ensure_initialized(self) -> None:
    #     """Lazy initialization of the agent."""
    #     if self.agent is None:
    #         self.agent = await ProviderAgent().initialize()
    
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        # await self._ensure_initialized()
        
        prompt = context.get_user_input()
        response = await self.agent.connect_to_server_and_run(prompt)
        await event_queue.enqueue_event(new_agent_text_message(response))
    
    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

def main():
    print("Running Healthcare Provider Agent")

    
    HOST = "localhost"
    PORT = 9997
    
    skill = AgentSkill(
        id="find_healthcare_providers",
        name="Find Healthcare Providers",
        description="Finds and lists healthcare providers based on user's location and specialty.",
        tags=["healthcare", "providers", "doctor", "psychiatrist"],
        examples=[
            "Are there any Psychiatrists near me in Boston, MA?",
            "Find a pediatrician in Springfield, IL.",
        ],
    )
    
    agent_card = AgentCard(
        name="HealthcareProviderAgent",
        description="An agent that can find and list healthcare providers based on a user's location and desired specialty.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )
    
    request_handler = DefaultRequestHandler(
        agent_executor=ProviderAgentExecutor(),
        task_store=InMemoryTaskStore(),
    )
    
    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )
    
    uvicorn.run(server.build(), host=HOST, port=PORT)
    
if __name__ == "__main__":
    main()

## Running the Arxiv Research Agent

- Open a terminal
- Navigate to the `1-WrapChatBotA2A` directory:
    - `cd 1-WrapChatBotA2A`
    - `uv init`
- Activate the virtual environment:
    - `uv venv`
    - `source .venv/bin/activate`
- Install the additional dependencies:
    - `uv add "a2a-sdk<1.0" mcp openai requests nest-asyncio`
- Run the chatbot:
    - `uv run arxiv_research_agent.py`
- To exit the chatbot, type `quit`.